# FRB path analysis with tav-resonance

This notebook demonstrates the **FRB × cosmic-web** resonance pipeline:

1. Pre-register tunable thresholds with `TavResonanceConfig`
2. Classify sightlines as `void_center`, `filament`, or `boundary_cross`
3. Compute DM residuals and search for **1/7 Tav harmonic** structure

Geometric anchors (313.1 MeV, 1/7 mode, α = 41.341) are **fixed** and never fitted.

In [ ]:
import numpy as np
import pandas as pd

from tav_resonance import (
    TavResonanceConfig,
    analyze_dm_residuals,
    classify_paths,
    geometric_anchor_check,
    run_resonance_scan,
)

## Verify fixed geometric anchors

In [ ]:
anchor_report = geometric_anchor_check()
anchor_report

## Toy catalogs

Replace these DataFrames with `load_frb_catalog()` / `load_void_catalog()` when you have CSV files on disk.

In [ ]:
rng = np.random.default_rng(42)
n = 48

frb = pd.DataFrame(
    {
        "ra": rng.uniform(0, 360, n),
        "dec": rng.uniform(-10, 80, n),
        "DM": rng.uniform(200, 900, n),
    }
)

voids = pd.DataFrame(
    {
        "ra": [45.0, 180.0, 300.0],
        "dec": [30.0, 10.0, 55.0],
        "reff_mpc": [18.0, 22.0, 15.0],
        "z_void": [0.05, 0.06, 0.04],
    }
)

frb.head()

## Pre-register configuration and freeze

In [ ]:
cfg = TavResonanceConfig.from_defaults()
cfg.harmonic_phase_tolerance = 0.04
cfg.harmonic_min_peaks = 2
cfg.freeze()

print(f"Frozen at: {cfg.frozen_at}")
print(f"Fixed ladder α: {cfg.ladder_alpha}")

## Path classification

In [ ]:
labels, separations = classify_paths(
    frb["ra"].to_numpy(),
    frb["dec"].to_numpy(),
    voids,
    config=cfg,
)

path_summary = pd.Series(labels, name="path_type").value_counts()
path_summary

## Full resonance scan

In [ ]:
result = run_resonance_scan(frb, voids, config=cfg)
print("\n".join(result.summary_lines()))

In [ ]:
result.frame[["ra", "dec", "DM", "path_type", "dm_residual"]].head(10)

In [ ]:
result.group_stats

## Optional: path-type bar chart

Requires `matplotlib` (`pip install tav-resonance[viz]`).

In [ ]:
try:
    import matplotlib.pyplot as plt

    counts = result.frame["path_type"].value_counts()
    counts.plot(kind="bar", title="FRB sightline path mix", ylabel="count")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skip plot or install tav-resonance[viz]")